In [6]:
import pandas as pd
from jiwer import wer, cer
from utils.num_to_words import numbers_to_words
import re
import evaluate

from comet import download_model, load_from_checkpoint


In [7]:
# Imports plot

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ast

# Setup

In [8]:
def clean_text(text):
    text = re.sub(r"-", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = text.lower()
    return text

comet = evaluate.load('comet')

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 61862.89it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.1.post0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/Users/claralouisebrodt/anaconda3/envs/02466_AI_dubbing/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [9]:
speakers = [
    "speaker_1", "speaker_2", "speaker_3", "speaker_4", "speaker_5",
    "speaker_6", "speaker_7", "speaker_8", "speaker_9", "speaker_10"
]

chunk_sizes = [3000, 3500, 4000, 4500, 5000, 5500, 6000]

# Data Preprocessing 

In [10]:

# Load COMET model once
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)



def data_results(input_language, output_language):
    df = pd.DataFrame(columns=['language','chunk_size', 'speaker', 'wer', 'cer', 'comet-score', 'stt_time', 'tt_time', 'tts_time', 'total_time'])
    
    
    for size in chunk_sizes:
        for speaker in speakers:
            # Load data
            if input_language == 'en':
                csv_df = pd.read_csv(f'logs/{speaker}_final_chunk{size}.csv')
            if input_language == 'da':
                csv_df = pd.read_csv(f'logs/dk_{speaker}_chunk{size}.csv')


            model_transcription = ' '.join(csv_df['transcription'].astype(str))
            model_translation = ' '.join(csv_df['translation'].astype(str))

            if input_language == 'en':
                with open(f"data/english/{speaker}_final.txt", "r", encoding="utf-8") as f:
                    ref_transcription = f.read()
                with open(f"data/danish/dk_{speaker}_final.txt", "r", encoding="utf-8") as f:
                    ref_translation = f.read()

            if input_language == 'da':
                with open(f"data/danish/dk_{speaker}_final.txt", "r", encoding="utf-8") as f:
                    ref_transcription = f.read()
                with open(f"data/english/{speaker}_final.txt", "r", encoding="utf-8") as f:
                    ref_translation = f.read()



            # Clean and normalize text
            model_transcription = clean_text(model_transcription)
            model_transcription = numbers_to_words(model_transcription, input_language, split_abbreviations=False)

            model_translation = clean_text(model_translation)
            model_translation = numbers_to_words(model_translation, output_language, split_abbreviations=False)

            ref_transcription = clean_text(ref_transcription)
            ref_transcription = numbers_to_words(ref_transcription, input_language, split_abbreviations=False)

            ref_translation = clean_text(ref_translation)
            ref_translation = numbers_to_words(ref_translation, output_language, split_abbreviations=False)

            if input_language == 'da':
                model_transcription = model_transcription.replace('øøhhm', '')
                model_translation = model_translation.replace('uhhm', '')

            # Compute WER and CER
            wer_score = wer(reference=ref_transcription, hypothesis=model_transcription)
            cer_score = cer(reference=ref_transcription, hypothesis=model_transcription)

            # Compute COMET score using direct model
            data = [{
                "src": model_transcription,
                "mt": model_translation,
                "ref": ref_transcription
            }]
            comet_score = comet_model.predict(data, batch_size=1, gpus=0, num_workers=1).scores[0]

            # Collect latency metrics
            stt_times = csv_df['stt_latency_ms'].tolist()
            tt_times = csv_df['tt_latency_ms'].tolist()
            tts_times = csv_df['tts_latency_ms'].tolist()
            total_times = csv_df['total_latency_ms'].tolist()

            # Append row to DataFrame
            df.loc[len(df)] = [
                input_language, size, speaker, wer_score, cer_score, comet_score,
                stt_times, tt_times, tts_times, total_times
            ]

    # Specify the folder path
    folder_path = 'results'  

    # Make sure the folder exists
    import os
    os.makedirs(folder_path, exist_ok=True)

    # Save the file in the specified folder
    df.to_csv(os.path.join(folder_path, f'results_{input_language}.csv'), index=False)


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 74631.74it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.1.post0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.


# Create Dataset

In [11]:
data_results('en', 'da')

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/claralouisebrodt/anaconda3/envs/02466_AI_dubbing/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
Predicting: 0it [00:00, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Predicting DataLoader 0: 100%|██████████| 1/1 [00:01<00:00,  1.67s/it]
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCh

In [12]:
data_results('da', 'en')

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/claralouisebrodt/anaconda3/envs/02466_AI_dubbing/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
Predicting DataLoader 0: 100%|██████████| 1/1 [00:01<00:00,  1.85s/it]
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/claralouisebrodt/anaconda3/envs/02466_AI_dubbing/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doi

# Data Visualization

In [21]:
df_en = pd.read_csv("results/results_en.csv")

df_en

,language,chunk_size,speaker,wer,cer,comet-score,stt_time,tt_time,tts_time,total_time
0,en,3000,speaker_1,0.181435,0.110599,0.524654,"[638.95, 629.3, 585.22, 663.36, 659.98, 538.74...","[770.81, 1331.01, 1065.7, 1177.26, 1423.59, 11...","[19503.35, 8506.92, 2982.81, 6891.27, 4187.45,...","[27317.51, 32159.37, 32073.88, 35904.86, 36823..."
1,en,3000,speaker_2,0.134483,0.105061,0.611333,"[569.82, 527.66, 492.73, 543.86, 533.52, 543.2...","[431.59, 803.65, 926.18, 922.3, 954.86, 923.66...","[4651.49, 6210.82, 5450.65, 2899.48, 3043.65, ...","[10065.51, 13214.37, 15399.93, 15235.18, 15214..."
2,en,3000,speaker_3,0.182109,0.122665,0.552829,"[531.35, 478.11, 454.33, 532.8, 486.24, 475.44...","[654.73, 816.95, 710.56, 1065.77, 1154.63, 950...","[6668.78, 2018.18, 2090.17, 5009.15, 2487.47, ...","[14445.03, 12983.84, 12014.18, 13548.88, 12767..."
3,en,3000,speaker_4,0.199396,0.131535,0.559549,"[502.36, 572.12, 499.81, 487.0, 489.63, 539.9,...","[536.83, 947.79, 848.8, 909.1, 878.91, 852.0, ...","[6287.88, 6187.89, 3186.4, 3647.08, 2412.84, 2...","[11823.01, 14911.5, 14961.92, 15507.69, 14810...."
4,en,3000,speaker_5,0.088000,0.077403,0.619074,"[548.58, 515.2, 535.76, 587.0, 528.67, 560.73,...","[563.51, 916.41, 1107.92, 1120.97, 799.52, 880...","[8055.59, 5625.49, 6612.3, 3048.43, 2668.77, 2...","[13859.31, 16382.33, 19876.42, 19611.81, 19148..."
...,...,...,...,...,...,...,...,...,...,...
65,en,6000,speaker_6,0.068354,0.046146,0.661043,"[673.17, 4490.49, 628.06, 632.93, 805.04, 703....","[1573.73, 2103.3, 2017.86, 1752.77, 2263.83, 2...","[23159.88, 10658.27, 6128.76, 6279.76, 7242.63...","[26040.39, 30562.92, 30567.76, 30111.82, 31233..."
66,en,6000,speaker_7,0.098266,0.071749,0.597477,"[617.82, 657.94, 541.01, 632.97, 704.94, 539.6...","[1565.28, 1664.62, 1419.12, 1557.24, 1754.02, ...","[18285.89, 5723.28, 4166.65, 5116.7, 9402.78, ...","[21163.0, 20660.16, 18179.51, 17060.14, 20057...."
67,en,6000,speaker_8,0.100000,0.074468,0.680271,"[701.94, 664.49, 624.89, 579.5, 581.41, 615.02...","[1796.13, 1613.85, 1372.85, 1446.4, 1245.17, 1...","[17435.98, 4227.13, 5180.44, 5100.95, 4028.47,...","[20727.89, 18747.92, 17696.38, 16351.98, 14210..."
68,en,6000,speaker_9,0.101333,0.063873,0.590529,"[785.82, 784.0, 784.12, 663.31, 722.91, 714.19...","[1416.36, 1958.33, 2212.42, 2019.92, 2021.99, ...","[19928.43, 6059.72, 21221.12, 5431.76, 7797.83...","[22515.43, 22347.58, 37132.18, 35696.34, 37261..."


In [22]:
df_da = pd.read_csv("results/results_en.csv")

df_da

,language,chunk_size,speaker,wer,cer,comet-score,stt_time,tt_time,tts_time,total_time
0,en,3000,speaker_1,0.181435,0.110599,0.524654,"[638.95, 629.3, 585.22, 663.36, 659.98, 538.74...","[770.81, 1331.01, 1065.7, 1177.26, 1423.59, 11...","[19503.35, 8506.92, 2982.81, 6891.27, 4187.45,...","[27317.51, 32159.37, 32073.88, 35904.86, 36823..."
1,en,3000,speaker_2,0.134483,0.105061,0.611333,"[569.82, 527.66, 492.73, 543.86, 533.52, 543.2...","[431.59, 803.65, 926.18, 922.3, 954.86, 923.66...","[4651.49, 6210.82, 5450.65, 2899.48, 3043.65, ...","[10065.51, 13214.37, 15399.93, 15235.18, 15214..."
2,en,3000,speaker_3,0.182109,0.122665,0.552829,"[531.35, 478.11, 454.33, 532.8, 486.24, 475.44...","[654.73, 816.95, 710.56, 1065.77, 1154.63, 950...","[6668.78, 2018.18, 2090.17, 5009.15, 2487.47, ...","[14445.03, 12983.84, 12014.18, 13548.88, 12767..."
3,en,3000,speaker_4,0.199396,0.131535,0.559549,"[502.36, 572.12, 499.81, 487.0, 489.63, 539.9,...","[536.83, 947.79, 848.8, 909.1, 878.91, 852.0, ...","[6287.88, 6187.89, 3186.4, 3647.08, 2412.84, 2...","[11823.01, 14911.5, 14961.92, 15507.69, 14810...."
4,en,3000,speaker_5,0.088000,0.077403,0.619074,"[548.58, 515.2, 535.76, 587.0, 528.67, 560.73,...","[563.51, 916.41, 1107.92, 1120.97, 799.52, 880...","[8055.59, 5625.49, 6612.3, 3048.43, 2668.77, 2...","[13859.31, 16382.33, 19876.42, 19611.81, 19148..."
...,...,...,...,...,...,...,...,...,...,...
65,en,6000,speaker_6,0.068354,0.046146,0.661043,"[673.17, 4490.49, 628.06, 632.93, 805.04, 703....","[1573.73, 2103.3, 2017.86, 1752.77, 2263.83, 2...","[23159.88, 10658.27, 6128.76, 6279.76, 7242.63...","[26040.39, 30562.92, 30567.76, 30111.82, 31233..."
66,en,6000,speaker_7,0.098266,0.071749,0.597477,"[617.82, 657.94, 541.01, 632.97, 704.94, 539.6...","[1565.28, 1664.62, 1419.12, 1557.24, 1754.02, ...","[18285.89, 5723.28, 4166.65, 5116.7, 9402.78, ...","[21163.0, 20660.16, 18179.51, 17060.14, 20057...."
67,en,6000,speaker_8,0.100000,0.074468,0.680271,"[701.94, 664.49, 624.89, 579.5, 581.41, 615.02...","[1796.13, 1613.85, 1372.85, 1446.4, 1245.17, 1...","[17435.98, 4227.13, 5180.44, 5100.95, 4028.47,...","[20727.89, 18747.92, 17696.38, 16351.98, 14210..."
68,en,6000,speaker_9,0.101333,0.063873,0.590529,"[785.82, 784.0, 784.12, 663.31, 722.91, 714.19...","[1416.36, 1958.33, 2212.42, 2019.92, 2021.99, ...","[19928.43, 6059.72, 21221.12, 5431.76, 7797.83...","[22515.43, 22347.58, 37132.18, 35696.34, 37261..."


## Plot for each STT, TT, TTS showing each author

In [14]:

def plot_latency_metrics_by_speaker(dataset):
    # Load CSV and parse list columns
    df = pd.read_csv(dataset)

    # Convert stringified lists to actual lists
    for col in ['stt_time', 'tt_time', 'tts_time']:
        df[col] = df[col].apply(ast.literal_eval)

    # Get language from first row
    language = df['language'].iloc[0].upper()  # Capitalize for nice display

    # Define chunk sizes
    chunk_sizes = [3000, 3500, 4000, 4500, 5000, 5500, 6000]

    # Explode helper
    def explode_time_by_speaker(df, time_col, chunk_sizes):
        records = []
        for _, row in df.iterrows():
            times = row[time_col]
            speaker = row['speaker']
            for i, time in enumerate(times):
                if i < len(chunk_sizes):
                    records.append({
                        'speaker': speaker,
                        'chunk_size': chunk_sizes[i],
                        'time': time
                    })
        return pd.DataFrame(records)

    # Explode time columns
    df_stt = explode_time_by_speaker(df, 'stt_time', chunk_sizes)
    df_tt = explode_time_by_speaker(df, 'tt_time', chunk_sizes)
    df_tts = explode_time_by_speaker(df, 'tts_time', chunk_sizes)

    # Set seaborn style
    sns.set(style="whitegrid")

    # STT Plot
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=df_stt, x='chunk_size', y='time', hue='speaker', marker='o')
    plt.ylim(0, 3500)  # Set Y-axis range for STT
    plt.title(f'STT Time per Speaker vs Chunk Size ({language})')
    plt.ylabel('STT Time (ms)')
    plt.xlabel('Chunk Size')
    plt.legend(title='Speaker', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # TT Plot
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=df_tt, x='chunk_size', y='time', hue='speaker', marker='o')
    plt.ylim(400, 2000)  # Set Y-axis range for TT
    plt.title(f'TT Time per Speaker vs Chunk Size ({language})')
    plt.ylabel('TT Time (ms)')
    plt.xlabel('Chunk Size')
    plt.legend(title='Speaker', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # TTS Plot
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=df_tts, x='chunk_size', y='time', hue='speaker', marker='o')
    plt.ylim(1800, 20000)  # Set Y-axis range for TTS
    plt.title(f'TTS Time per Speaker vs Chunk Size ({language})')
    plt.ylabel('TTS Time (ms)')
    plt.xlabel('Chunk Size')
    plt.legend(title='Speaker', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


### Plot English

In [ ]:
plot_latency_metrics_by_speaker("results/results_en.csv")

### Plot English

In [ ]:
plot_latency_metrics_by_speaker("results/results_da.csv")

## Plot for each STT, TT, TTS showing average

In [15]:
def plot_avg_latency_by_chunk_size(dataset):
    # Load CSV
    df = pd.read_csv(dataset)


    # Get language from first row
    language = df['language'].iloc[0].upper()  # Capitalize for nice display

    # Convert stringified lists to actual lists
    for col in ['stt_time', 'tt_time', 'tts_time']:
        df[col] = df[col].apply(ast.literal_eval)

    # Compute average latencies per row
    df['avg_stt_time'] = df['stt_time'].apply(lambda x: sum(x) / len(x))
    df['avg_tt_time'] = df['tt_time'].apply(lambda x: sum(x) / len(x))
    df['avg_tts_time'] = df['tts_time'].apply(lambda x: sum(x) / len(x))

    # Group by chunk size and compute mean across speakers
    grouped = df.groupby('chunk_size')[['avg_stt_time', 'avg_tt_time', 'avg_tts_time']].mean().reset_index()

    # Plot
    sns.set(style="whitegrid")
    plt.figure(figsize=(10, 6))

    sns.lineplot(data=grouped, x='chunk_size', y='avg_stt_time', marker='o', label='STT Time')
    sns.lineplot(data=grouped, x='chunk_size', y='avg_tt_time', marker='o', label='TT Time')
    sns.lineplot(data=grouped, x='chunk_size', y='avg_tts_time', marker='o', label='TTS Time')

    plt.title(f'Average STT, TT, and TTS Time vs Chunk Size ({language})')

    plt.ylim(0, 6500)  # Set Y-axis range for STT
    plt.ylabel('Time (s)')
    plt.xlabel('Chunk Size')
    plt.legend(title='Metric')
    plt.tight_layout()
    plt.show()


In [ ]:
plot_avg_latency_by_chunk_size("results/results_en.csv")

In [ ]:
plot_avg_latency_by_chunk_size("results/results_da.csv")

## Plot CER and WER

In [16]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_wer_cer_by_chunk_and_language(danish_csv, english_csv):
    # Load both CSVs
    df_da = pd.read_csv(danish_csv)
    df_en = pd.read_csv(english_csv)

    # Compute average WER and CER per chunk size
    grouped_da = df_da.groupby('chunk_size')[['wer', 'cer']].mean().reset_index()
    grouped_en = df_en.groupby('chunk_size')[['wer', 'cer']].mean().reset_index()

    # Set seaborn style
    sns.set(style="whitegrid")

    # Plot
    plt.figure(figsize=(10, 6))

    # Danish: dark blue for WER, light blue for CER
    sns.lineplot(data=grouped_da, x='chunk_size', y='wer', color='red', label='Danish WER', marker='o')
    sns.lineplot(data=grouped_da, x='chunk_size', y='cer', color='lightcoral', label='Danish CER', marker='o')

    # English: dark red for WER, light red for CER
    sns.lineplot(data=grouped_en, x='chunk_size', y='wer', color='navy', label='English WER', marker='o')
    sns.lineplot(data=grouped_en, x='chunk_size', y='cer', color='skyblue', label='English CER', marker='o')

    # Labels and legend
    plt.title('WER and CER per Chunk Size (Danish vs English)')
    plt.xlabel('Chunk Size')
    plt.ylabel('Error Rate')
    plt.legend(title='Metric')
    plt.tight_layout()
    plt.show()


In [ ]:
plot_wer_cer_by_chunk_and_language("results/results_da.csv", "results/results_en.csv")

## Plot COMET

In [17]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_comet_by_chunk_and_language(danish_csv, english_csv):
    # Load both CSVs
    df_da = pd.read_csv(danish_csv)
    df_en = pd.read_csv(english_csv)

    # Compute average WER and CER per chunk size
    grouped_da = df_da.groupby('chunk_size')[['comet-score']].mean().reset_index()
    grouped_en = df_en.groupby('chunk_size')[['comet-score']].mean().reset_index()

    # Set seaborn style
    sns.set(style="whitegrid")

    # Plot
    plt.figure(figsize=(10, 6))


    sns.lineplot(data=grouped_da, x='chunk_size', y='comet-score', color='red', label='Danish to English', marker='o')

    sns.lineplot(data=grouped_en, x='chunk_size', y='comet-score', color='navy', label='English to Danish', marker='o')

    # Labels and legend
    plt.title('Comet-score per Chunk Size (Danish vs English)')
    plt.xlabel('Chunk Size')
    plt.ylabel('Comet Score')
    plt.legend(title='Metric')
    plt.tight_layout()

    plt.show()


In [ ]:
plot_comet_by_chunk_and_language("results/results_da.csv", "results/results_en.csv")

## Save all plots to folder

In [20]:
import os
import matplotlib.pyplot as plt

# Ensure the "images" folder exists
os.makedirs("images", exist_ok=True)

# Backup the original plt.show function
_original_show = plt.show

# Counter to give unique filenames
_plot_counter = 0

def save_figs_to_folder():
    # Override plt.show to save plots instead of displaying them.
    def _save_and_close(*args, **kwargs):
        global _plot_counter
        filename = f"images/plot_{_plot_counter:02d}.png"
        plt.savefig(filename, bbox_inches='tight')
        plt.close()
        _plot_counter += 1
    plt.show = _save_and_close

def restore_show():
    # Restore the original plt.show behavior.
    plt.show = _original_show

# Apply the override
save_figs_to_folder()

# Call your functions as-is
plot_latency_metrics_by_speaker("results/results_en.csv")
plot_latency_metrics_by_speaker("results/results_da.csv")
plot_avg_latency_by_chunk_size("results/results_en.csv")
plot_avg_latency_by_chunk_size("results/results_da.csv")
plot_wer_cer_by_chunk_and_language("results/results_da.csv", "results/results_en.csv")
plot_comet_by_chunk_and_language("results/results_da.csv", "results/results_en.csv")

# Restore normal behavior if needed
restore_show()
